# 0. Autores
### Mauro Virgilio Blanc
### Juan Pablo Imbrogno
### Sofía Belén Caselli
### Miguel Angel Leiva Martinez
### Andrea Viviana Ferenaz

# 1. Introducción

El problema abordado corresponde a una **tarea de clasificación supervisada**, cuyo objetivo es predecir la probabilidad de que un paciente sufra un accidente cerebrovascular (ACV) a partir de variables demográficas y clínicas (edad, hipertensión, nivel de glucosa, tipo de trabajo, tabaquismo, entre otras).  

La variable objetivo `stroke` es **binaria** (0 = no tuvo ACV, 1 = tuvo ACV) y presenta un fuerte **desbalance de clases** (~5% positivos). Por ello, se priorizaron algoritmos robustos frente a este tipo de distribución y se aplicaron técnicas de *oversampling* (SMOTE) durante el entrenamiento para mejorar la capacidad de detección de la clase minoritaria.

# 2. Selección del Algoritmo

Se entrenaron múltiples modelos representativos de diferentes enfoques de aprendizaje supervisado:

| Modelo                  | Descripción breve                                           |
| ----------------------- | ----------------------------------------------------------- |
| **Regresión Logística** | Modelo lineal interpretable, base de referencia.            |
| **Random Forest**       | Árboles de decisión con *bagging*, robusto frente a ruido.  |
| **XGBoost**             | Boosting secuencial, captura relaciones no lineales.        |
| **CatBoost**            | Boosting eficiente con variables categóricas.               |
| **SVM**                 | Separación por márgenes, útil para datasets desbalanceados. |
| **KNN**                 | Basado en vecinos, sensible al escalado.                    |
| **Soft Voting**         | Ensamble de modelos mediante promedio de probabilidades.    |

# 3. Resultados
### 3.1 Comparación de Modelos
   
| Modelo              | AUC promedio | Desvío estándar |
| ------------------- | ------------ | --------------- |
| Logistic Regression | **0.834**    | 0.0186          |
| Soft Voting         | 0.812        | 0.0155          |
| Random Forest       | 0.788        | 0.0143          |
| CatBoost            | 0.787        | 0.0184          |
| XGBoost             | 0.807        | 0.0140          |
| SVM                 | 0.755        | 0.0227          |
| KNN                 | 0.701        | 0.0170          |


### 3.2 Interpretación:
- La **Regresión Logística** obtuvo el AUC más alto (≈0.834), indicando que es el modelo con mejor capacidad de discriminación en este conjunto de datos.  
- SoftVoting, aunque ligeramente inferior en AUC (≈0.812), combina la fuerza de varios clasificadores, lo que puede aportar **robustez frente a distintos tipos de datos**.  
- Modelos complejos como XGBoost y CatBoost no superaron a la Regresión Logística en esta etapa, probablemente debido a la cantidad limitada de datos positivos o a la necesidad de mayor ajuste de hiperparámetros.

### 3.3 Selección y optimización

- Se seleccionó **Regresión Logística** como modelo principal por su **AUC superior**, simplicidad e interpretabilidad.  
- Se aplicó **SMOTE** para balancear la clase minoritaria durante el entrenamiento.  
- Posteriormente, se optimizaron los hiperparámetros mediante ajuste de **C y penalización L1**, logrando mejorar la capacidad de detección de la clase positiva.

### 3.4 Métricas en conjunto de prueba (aproximadas):

- Accuracy: 0.77–0.78  
- Precision: 0.15–0.16  
- Recall: 0.64–0.80  
- F1-score: 0.26  
- ROC-AUC: 0.83  

### 3.5 Interpretación:
- El modelo muestra buena capacidad de discriminación y logra detectar la mayoría de los casos positivos, lo cual es crítico en un contexto médico donde los falsos negativos tienen alto costo.  
- La precisión baja refleja el desbalance de clases, pero el recall alto asegura que la mayoría de eventos positivos sean identificados.

### 3.6 Umbral de Decisión Óptimo y Matriz de Confusión Clínica

Aunque el modelo de Regresión Logística optimizado con SMOTE fue seleccionado como el más adecuado por su **AUC superior** ($\approx 0.834$) y demostró una buena capacidad de discriminación, la implementación en un contexto clínico debe priorizar la minimización de **Falsos Negativos** (FNs). Esto es fundamental, ya que el problema abordado es una **tarea de clasificación supervisada** con un **fuerte desbalance de clases** ($\sim 5\%$ positivos).

1.  **Umbral de Decisión Estándar (0.5):**
    *   Al usar el umbral por defecto de $0.5$ en el conjunto de prueba, se registraron **10 Falsos Negativos** (ACV no detectado) y **40 Verdaderos Positivos** (ACV detectado).

2.  **Umbral de Decisión Optimizado:**
    *   Para mejorar el *screening* y reducir el riesgo clínico asociado a los FNs, se determinó un **umbral de decisión ajustado** que maximiza el *recall*.
    *   El valor del umbral óptimo calculado fue de **$0.17847$** .

Al aplicar este umbral de decisión ajustado, la matriz de confusión resultante en el conjunto de prueba demuestra una mejora crítica en la capacidad de detección, **reduciendo los Falsos Negativos a 5**:

| Real / Predicho | No ACV (0) | ACV (1) |
| :---: | :---: | :---: |
| **No ACV (0)** | 465 | 507 |
| **ACV (1)** | **5** | **45** |

Esta modificación del umbral es esencial para la utilidad práctica del modelo en salud, asegurando que se identifique la mayor parte de los eventos positivos (ACV).

# 4. Importancia de las variables

El análisis de los coeficientes de la **Regresión Logística** indicó que las variables más influyentes fueron:

*   **Edad** 
*   **Tipo de residencia (urbana/rural)**  
*   **Hipertensión**  
*   **Tabaquismo**  
*   **Índice de masa corporal (BMI)**  
*   **Nivel de glucosa promedio**  
*   **Tipo de trabajo**  

Estas variables coinciden con la evidencia clínica sobre los factores de riesgo asociados al ACV, reforzando la validez del modelo.

# 5. Conclusión

El modelo final, **Regresión Logística optimizado con SMOTE**, fue elegido como el más adecuado para esta **tarea de clasificación supervisada** por su excelente balance entre **rendimiento (AUC $\approx 0.83$), interpretabilidad y estabilidad**.

A pesar del fuerte desbalance de clases del *dataset* ($\sim 5\%$ positivos), el desempeño del modelo fue satisfactorio para el objetivo de trabajo, logrando una alta capacidad de detección de la clase positiva (alto *recall*).

Como directrices para la implementación práctica, se debe:
1.  Utilizar el modelo de Regresión Logística, dada su interpretabilidad clínica.
2.  Implementar el **ajuste del umbral de decisión** (alrededor de $0.17847$) para minimizar los Falsos Negativos y priorizar el *screening* de riesgo.

In [ ]:
# Instalación de dependencias
!pip install catboost imbalanced-learn -q

In [ ]:
# ===========================================
# TRABAJO PRÁCTICO - LENGUAJE MÁQUINA
# Predicción de ACV (Stroke Prediction Dataset)
# ===========================================

# === Librerías estándar ===
import warnings

# === Análisis y manipulación ===
import numpy as np
import pandas as pd

# === Visualización ===
import matplotlib.pyplot as plt
import seaborn as sns

# === Preprocesamiento ===
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# === Modelos ===
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Imbalanced learning ===
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# === Evaluación y selección ===
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split

# === Configuración global ===
RANDOM_STATE = 42
DATA_PATH = "https://raw.githubusercontent.com/maurovirgilioblanc-cmd/TP_MLOP/main/data/stroke-data.csv"
TEST_SIZE = 0.20
CV_FOLDS = 5
RECALL_TARGET = 0.90

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100})

In [ ]:
# ===========================================
# CARGA Y EXPLORACIÓN DEL DATASET
# ===========================================

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

print("\nInformación del dataset:")
print(df.info())

# Valores únicos por columna
n_nunique = df.nunique()
print("\nValores únicos por columna:")
print(n_nunique)

# Duplicados
n_duplicates = df.duplicated().sum()
print(f"\nFilas duplicadas: {n_duplicates}")

# Valores nulos
aux = df.isna().mean().sort_values(ascending=False)
print("\nSi tiene >40-50% es ruido:")
print(aux)

nulls = df.isnull().sum()
print("\nValores nulos por columna:")
print(nulls[nulls > 0] if nulls.sum() > 0 else "No hay valores nulos")

# Variables con mucha cardinalidad
cat_features = [col for col in df.columns if df[col].nunique() > 20 and df[col].dtype != 'float64']
print("\nVariables con mucha cardinalidad:")
print(cat_features)

# Varianza
numeric_cols = df.select_dtypes(include=['float64', 'int64'])
selector = VarianceThreshold(threshold=0.01)
selector.fit(numeric_cols)
cols_quedan = numeric_cols.columns[selector.get_support()].tolist()
cols_eliminadas = numeric_cols.columns[~selector.get_support()].tolist()
print("\nCOLUMNAS QUE QUEDAN:", cols_quedan)
print("COLUMNAS ELIMINADAS POR BAJA VARIANZA:", cols_eliminadas)

# Correlación
print("\nANALIZO CORRELACION SOLO NUMERICAS:")
correrl = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(correrl, annot=True, fmt=".2f", cmap="coolwarm", ax=ax, linewidths=0.5)
ax.set_title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()

# Distribuciones categóricas
print("\nAnalizo Distribuciones:")
for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
    print(f"\n{col}:")
    print(df[col].value_counts(normalize=True))

# Distribución variable objetivo
print("\nDistribución de 'stroke':")
print(df['stroke'].value_counts(normalize=True).rename({0: "No ACV", 1: "ACV"}))

stroke_rate = df['stroke'].mean()
print(f"\nPrevalencia real de ACV: {stroke_rate:.4f} ({stroke_rate*100:.2f}%)")

df['stroke'].value_counts().plot(kind='bar')
plt.title("Distribución de la variable objetivo (stroke)")
plt.xticks([0, 1], ["No ACV", "ACV"], rotation=0)
plt.ylabel("Cantidad")
plt.show()

In [ ]:
# ===========================================
# LIMPIEZA DE DATOS
# ===========================================
df = df.drop(columns=['id'])  # ID no aporta valor predictivo

# ===========================================
# NUEVAS VARIABLES DERIVADAS (FEATURE ENGINEERING)
# ===========================================

# Edad categorizada
df['age_group'] = pd.cut(df['age'], bins=[0, 30, 50, 70, 120], labels=['Joven', 'Adulto', 'Mayor', 'Anciano'])
df['age_group'] = df['age_group'].astype(str)

# Normalizar valores extremos de glucosa
df['avg_glucose_level'] = np.clip(df['avg_glucose_level'], 50, 300)

# Riesgo combinado de hipertensión y enfermedad cardíaca
df['has_risk_factors'] = np.where((df['hypertension'] == 1) | (df['heart_disease'] == 1), 1, 0)

# Codificación binaria del género
# Other 0.000196 queda dentro de la mayoritaria (Female)
df['is_female'] = np.where(df['gender'] == 'Female', 1, 0)
df = df.drop(columns=['gender'])

# Nunca trabajo (0.4%) lo uno con children (13.4%) - representan mismo grupo
df["work_type"] = df["work_type"].replace({"Never_worked": "children"})
# Según neurólogos estos no son factores de riesgo
# df = df.drop(columns=['ever_married','work_type','Residence_type'])

In [ ]:
# ===========================================
# SEPARAR VARIABLES Y CONJUNTOS DE DATOS
# ===========================================
X = df.drop('stroke', axis=1)
y = df['stroke']

# División estratificada: 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# Completar nulos DESPUÉS del split para evitar leakage
# La mediana se calcula solo sobre X_train para no filtrar info del test
bmi_median = X_train['bmi'].median()
X_train = X_train.copy()
X_test = X_test.copy()
X_train['bmi'] = X_train['bmi'].fillna(bmi_median)
X_test['bmi'] = X_test['bmi'].fillna(bmi_median)

In [ ]:
# ===========================================
# PREPROCESAMIENTO
# ===========================================

CAT_FEATURES = ['smoking_status', 'age_group']
NUM_FEATURES = X.select_dtypes(exclude='object').columns.drop('is_female').tolist()

def build_preprocessor(num_cols=None, cat_cols=None):
    """Construye un ColumnTransformer con StandardScaler + OneHotEncoder."""
    if num_cols is None:
        num_cols = NUM_FEATURES
    if cat_cols is None:
        cat_cols = CAT_FEATURES
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
        ],
        remainder='drop'
    )

# Preprocessor genérico
preprocessor = build_preprocessor()

# Preprocessor para Logistic Regression (sin columnas ruidosas)
# heart_disease, bmi, smoking_status, age_group → coef=0 con L1 → ruido
cols_eliminar_LR = ["heart_disease", "bmi", "smoking_status", "age_group"]
num_features_lr = [c for c in NUM_FEATURES if c not in cols_eliminar_LR]
cat_features_lr = [c for c in CAT_FEATURES if c not in cols_eliminar_LR]
preprocessor_lr = build_preprocessor(num_features_lr, cat_features_lr)

In [ ]:
# ===========================================
# Evalúo variables ruidosas en Regresión Logística
# ===========================================

def get_lr_feature_importance(pipe, num_cols, cat_cols):
    """Extrae los coeficientes de una Regresión Logística dentro de un Pipeline."""
    ohe = pipe.named_steps['preprocessor'].named_transformers_['cat']
    ohe_features = ohe.get_feature_names_out(cat_cols).tolist()
    all_features = num_cols + ohe_features
    coefs = pipe.named_steps['model'].coef_[0]
    return (
        pd.DataFrame({'feature': all_features, 'coef': coefs, 'abs_coef': np.abs(coefs)})
        .sort_values('abs_coef', ascending=False)
        .reset_index(drop=True)
    )

# L1 fuerza coefs cero en variables ruidosas
lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=2000,
        penalty='l1',
        solver='liblinear',
        class_weight='balanced',
        C=0.1
    ))
])
lr_model.fit(X_train, y_train)

importancia_lr = get_lr_feature_importance(lr_model, NUM_FEATURES, CAT_FEATURES)
display(importancia_lr)

# heart_disease → coef=0 → pocos positivos, no logra aprender
# bmi → coef=0 → relación débil con stroke en datasets chicos
# smoking_status_formerly → coef=0 → no aporta diferencia
# age_group_Anciano → coef=0 → age numérica ya explica todo

In [ ]:
# ===========================================
# Evalúo variables ruidosas en XGBoost
# ===========================================

pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        scale_pos_weight=pos_weight,
        use_label_encoder=False,
        colsample_bytree=0.7,
        gamma=0.3,
        learning_rate=0.01,
        max_depth=3,
        min_child_weight=1,
        n_estimators=100,
        subsample=1.0
    ))
])
xgb_pipe.fit(X_train, y_train)

ohe = xgb_pipe.named_steps['preprocessor'].named_transformers_['cat']
ohe_cols = ohe.get_feature_names_out(CAT_FEATURES).tolist()
all_features_xgb = NUM_FEATURES + ohe_cols

importancia_xgb = (
    pd.DataFrame({
        'feature': all_features_xgb,
        'importance': xgb_pipe.named_steps['model'].feature_importances_
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
print(importancia_xgb)

features_to_drop = importancia_xgb[importancia_xgb['importance'] < 0.01]['feature'].tolist()
print("Features con baja importancia en XGB:", features_to_drop)

In [ ]:
def categorizar_riesgo(prob_array):
    """
    Categoriza probabilidades en Bajo / Medio / Alto según percentiles 70 y 90.

    Parameters
    ----------
    prob_array : np.ndarray
        Array de probabilidades de la clase positiva (valores en [0, 1]).

    Returns
    -------
    np.ndarray
        Etiqueta de riesgo para cada observación.
    """
    p70 = np.percentile(prob_array, 70)
    p90 = np.percentile(prob_array, 90)
    result = np.full(len(prob_array), "Bajo", dtype=object)
    result[(prob_array >= p70) & (prob_array < p90)] = "Medio"
    result[prob_array >= p90] = "Alto"
    return result

In [ ]:
# ===========================================
# ENTRENAMIENTO DE MODELOS BASE
# ===========================================
# Se prueban múltiples modelos con SMOTE.
# cross_val_score con roc_auc valida la capacidad de distinguir ACV/no-ACV.

smote = SMOTE(random_state=RANDOM_STATE)

# Pipeline KNN (requiere escalado propio)
knn_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=7))
])

# Clasificadores base para VotingClassifier
models_base = [
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')),
    ("rf", RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced')),
    ("xgb", XGBClassifier(
        eval_metric='logloss', random_state=RANDOM_STATE,
        scale_pos_weight=float(pos_weight), use_label_encoder=False
    )),
    ("cat", CatBoostClassifier(verbose=0, random_state=RANDOM_STATE, scale_pos_weight=float(pos_weight))),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    ("knn", knn_model)
]

soft_voting = VotingClassifier(estimators=models_base, voting='soft')

# Catálogo: (nombre, modelo, preprocessor)
MODEL_CATALOG = [
    ("LogisticRegression",
     LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
     preprocessor_lr),
    ("RandomForest",
     RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
     preprocessor),
    ("XGBoost",
     XGBClassifier(
         eval_metric='logloss', random_state=RANDOM_STATE,
         scale_pos_weight=float(pos_weight), use_label_encoder=False,
         colsample_bytree=0.7, gamma=0.3, learning_rate=0.01,
         max_depth=3, min_child_weight=1, n_estimators=100, subsample=1.0
     ),
     preprocessor),
    ("CatBoost",
     CatBoostClassifier(verbose=0, random_state=RANDOM_STATE, scale_pos_weight=float(pos_weight)),
     preprocessor),
    ("SVM",
     SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
     preprocessor),
    ("KNN", knn_model, preprocessor),
    ("SoftVoting", soft_voting, preprocessor),
]

results = []
trained_pipes = {}
y_pred_proba = {}
y_riesgo_dict = {}

print("\n--- Modelos base con SMOTE ---")

for name, model, prep in MODEL_CATALOG:
    pipe = ImbPipeline(steps=[
        ('preprocessor', prep),
        ('smote', smote),
        ('model', model)
    ])

    scores = cross_val_score(pipe, X_train, y_train, cv=CV_FOLDS, scoring='roc_auc', n_jobs=-1)
    mean_auc = scores.mean()
    std_auc = scores.std()

    print(f"{name} + SMOTE AUC: mean={mean_auc:.4f}, std={std_auc:.4f}")
    results.append({"Modelo": name, "AUC promedio": mean_auc, "AUC std": std_auc})

    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe

    try:
        y_prob = pipe.predict_proba(X_test)[:, 1]
        y_pred_proba[name] = y_prob
        y_riesgo_dict[name] = categorizar_riesgo(y_prob)
    except AttributeError:
        print(f"{name} no soporta predict_proba y se salteará la curva ROC")

resultados_df = pd.DataFrame(results).sort_values("AUC promedio", ascending=False).reset_index(drop=True)
print("\n===== COMPARACIÓN DE MODELOS =====")
display(resultados_df)

In [ ]:
# ===========================================
# ANÁLISIS DE RIESGO + GRÁFICOS
# ===========================================

modelo = "LogisticRegression"

y_real = y_test
y_proba = y_pred_proba[modelo]
y_riesgo = categorizar_riesgo(y_proba)
y_pred_bin = (y_proba >= 0.5).astype(int)

df_results = pd.DataFrame({
    "Real": y_real,
    "Probabilidad": y_proba,
    "Predicho": y_pred_bin,
    "Riesgo": y_riesgo
})

print(df_results.head())

df_results["Caso"] = df_results["Real"].astype(str) + " - " + df_results["Predicho"].astype(str)
casos_count = df_results["Caso"].value_counts().sort_index()
print("\n=== Cantidad por caso (0-0, 0-1, 1-0, 1-1) ===")
print(casos_count)

# Gráfico 1: Distribución de riesgo
plt.figure(figsize=(6, 4))
df_results["Riesgo"].value_counts().reindex(["Bajo", "Medio", "Alto"]).plot(kind="bar")
plt.title("Distribución de Niveles de Riesgo")
plt.xlabel("Categoría de Riesgo")
plt.ylabel("Cantidad de Pacientes")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

# Gráfico 2: Casos
plt.figure(figsize=(6, 4))
casos_count.plot(kind="bar")
plt.title("Distribución de Casos (Real - Predicho)")
plt.xlabel("Caso")
plt.ylabel("Cantidad")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de riesgo con porcentajes
riesgos = y_riesgo_dict["LogisticRegression"]
counts = pd.Series(riesgos).value_counts(normalize=True) * 100

colores = {"Bajo": "green", "Medio": "orange", "Alto": "red"}

plt.figure(figsize=(7, 4))
bars = plt.bar(counts.index, counts.values, color=[colores.get(cat, "blue") for cat in counts.index])
plt.ylabel("Porcentaje (%)")
plt.title("Distribución de Riesgo según Logistic Regression")

for i, v in enumerate(counts.values):
    plt.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')

plt.ylim(0, max(counts.values) + 10)
plt.show()

In [ ]:
# Comparación de AUC entre modelos
best_idx = resultados_df["AUC promedio"].idxmax()
best_auc = resultados_df.loc[best_idx, "AUC promedio"]

plt.figure(figsize=(8, 4))
bars = plt.bar(resultados_df["Modelo"], resultados_df["AUC promedio"])

for i, bar in enumerate(bars):
    auc_val = resultados_df.loc[i, "AUC promedio"]
    if i == best_idx:
        bar.set_color("red")
        label = f"Mejor ({auc_val:.3f})"
    else:
        bar.set_color("blue")
        diff = best_auc - auc_val
        label = f"-{diff:.3f}"
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             label, ha='center', va='bottom', fontsize=10)

plt.ylim(0, best_auc + 0.1)
plt.title("Comparación de modelos base (AUC)")
plt.ylabel("AUC promedio")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Diferencia con el mejor modelo
best_auc = resultados_df["AUC promedio"].max()
diff = best_auc - resultados_df["AUC promedio"]

plt.figure(figsize=(8, 4))
plt.bar(resultados_df["Modelo"], diff, color='salmon')
plt.xticks(rotation=45)
plt.ylabel("Diferencia con el mejor AUC")
plt.title("Cuánto pierde cada modelo frente al mejor")
plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC de todos los modelos
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8, 6))
for modelo in y_pred_proba:
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba[modelo])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{modelo} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curvas ROC de los modelos')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluar Logistic Regression SIN SMOTE (comparación)
pipe_lr = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=CV_FOLDS, scoring='roc_auc')
print("LR sin SMOTE: AUC mean =", scores_lr.mean(), "std =", scores_lr.std())

In [ ]:
# ===========================================
# OPTIMIZACIÓN DE HIPERPARÁMETROS
# ===========================================

lr_grid = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.1, 1, 3, 10],
    "model__solver": ["liblinear"]  # necesario para L1
}

pipe_lr_gs = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
])

grid = GridSearchCV(pipe_lr_gs, lr_grid, cv=CV_FOLDS, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best LR params:", grid.best_params_)
print("Best LR AUC:", grid.best_score_)

In [ ]:
# Entrenar el modelo final con los mejores hiperparámetros
best_lr = grid.best_estimator_
best_lr.fit(X_train, y_train)

In [ ]:
# Evaluación en TEST
y_prob_final = best_lr.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_prob_final)
print("AUC en TEST:", auc_test)
# Si AUC en test es similar (ej: 0.83-0.84), está validado

In [ ]:
# Distribución de riesgo del modelo final
df_results_final = pd.DataFrame({
    "y_test": y_test,
    "prob": y_prob_final
})

df_results_final["riesgo"] = categorizar_riesgo(y_prob_final)

df_results_final["riesgo"].value_counts().reindex(["Bajo", "Medio", "Alto"]).plot(kind="bar")
plt.title("Distribución de niveles de riesgo")
plt.xlabel("Riesgo")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
# ===========================================
# UMBRAL ÓPTIMO PARA SCREENING CLÍNICO
# ===========================================

precision_arr, recall_arr, thresholds_arr = precision_recall_curve(y_test, y_prob_final)

# Threshold más alto que garantiza recall >= RECALL_TARGET
mask = recall_arr[:-1] >= RECALL_TARGET
best_threshold = float(thresholds_arr[mask].max()) if mask.any() else 0.5
print("Threshold óptimo para screening:", best_threshold)

y_pred_screening = (y_prob_final >= best_threshold).astype(int)

# Comparación de umbrales
y_pred_default = (y_prob_final >= 0.5).astype(int)

print("\n=== Umbral 0.50 ===")
print(classification_report(y_test, y_pred_default, target_names=["No ACV", "ACV"]))

print(f"=== Umbral {best_threshold:.3f} (optimizado) ===")
print(classification_report(y_test, y_pred_screening, target_names=["No ACV", "ACV"]))

# Matrices de confusión lado a lado
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, y_pred, title in zip(
    axes,
    [y_pred_default, y_pred_screening],
    ["Umbral estándar (0.50)", f"Umbral optimizado ({best_threshold:.3f})"],
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=["No ACV", "ACV"],
    ).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)

plt.suptitle("Comparación de matrices de confusión", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================
# GRÁFICO FINAL DE RIESGO Y CALIBRACIÓN
# ===========================================

CATEGORIAS = ["Bajo", "Medio", "Alto"]
COLORES = {"Bajo": "#2ca02c", "Medio": "#ff7f0e", "Alto": "#d62728"}

df_resultados = pd.DataFrame({
    "y_real": y_test.values,
    "probabilidad": y_prob_final,
    "pred_binaria": y_pred_screening,
    "riesgo": categorizar_riesgo(y_prob_final),
})

proporciones = df_resultados["riesgo"].value_counts(normalize=True).reindex(CATEGORIAS)
tasas_positivos = [
    df_resultados.loc[df_resultados["riesgo"] == nivel, "y_real"].mean()
    for nivel in CATEGORIAS
]

fig, ax1 = plt.subplots(figsize=(8, 6))

bars = ax1.bar(
    CATEGORIAS, proporciones.values,
    color=[COLORES[c] for c in CATEGORIAS], alpha=0.8, edgecolor="black"
)
ax1.set_ylabel("Proporción de pacientes")
ax1.set_xlabel("Categoría de riesgo")
ax1.set_ylim(0, 1)
ax1.set_title("Distribución de riesgo y tasa real de ACV", fontsize=14, fontweight="bold")

for bar, prop in zip(bars, proporciones.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
             f"{prop*100:.1f}%", ha='center', va='center', color="white", fontweight="bold")

ax2 = ax1.twinx()
ax2.plot(CATEGORIAS, tasas_positivos, marker='o', linestyle='--', linewidth=2, color='blue', label="Tasa real ACV")
ax2.set_ylabel("Tasa real de positivos (ACV)")
ax2.set_ylim(0, 1)

for i, t in enumerate(tasas_positivos):
    ax2.text(i, t + 0.03, f"{t*100:.1f}%", ha='center', color='blue', fontsize=10, fontweight="bold")

ax2.legend(loc="upper left")
plt.tight_layout()
plt.show()